In [7]:
import firebase_admin
from firebase_admin import credentials, db
import pandas as pd
from datetime import datetime
import os

# --- 1. SETTING PATH (SUDAH DISESUAIKAN DENGAN ISI FOLDERMU) ---
PATH_JSON = '/content/drive/MyDrive/SKRIPSHIT/firebase-project/firebase-project/serviceAccountKey.json'
DB_URL = 'https://test-reading-the-pzem-default-rtdb.asia-southeast1.firebasedatabase.app'

# --- 2. KONEKSI KE FIREBASE ---
if not firebase_admin._apps:
    if os.path.exists(PATH_JSON):
        cred = credentials.Certificate(PATH_JSON)
        firebase_admin.initialize_app(cred, {'databaseURL': DB_URL})
        print("✅ KONEKSI SUKSES: Firebase Berhasil Terhubung!")
    else:
        print("❌ ERROR: File serviceAccountKey.json tidak ditemukan di path tersebut.")
else:
    print("✅ STATUS: Firebase sudah aktif.")

# --- 3. PROSES PERBAIKAN DATA (KWH & STATE) ---
try:
    print("⏳ Sedang menghitung ulang data dari 'history'...")
    ref_hist = db.reference('history')
    raw = ref_hist.get()

    if raw:
        # Konversi ke DataFrame
        df = pd.DataFrame.from_dict(raw, orient='index')

        # Penyeragaman Waktu
        if 'waktu' in df.columns:
            df['waktu'] = pd.to_datetime(df['waktu'])
        else:
            # Jika hardware belum kirim waktu, buatkan estimasi mundur dari sekarang
            waktu_sekarang = datetime.now()
            df.index = pd.date_range(end=waktu_sekarang, periods=len(df), freq='10S')
            df['waktu'] = df.index

        # Pastikan Daya adalah angka (Numerik)
        df['Daya'] = pd.to_numeric(df['Daya'], errors='coerce').fillna(0)
        df.set_index('waktu', inplace=True)

        # RUMUS: Menghitung Wh (Watt-hour) dari data per 10 detik
        df['Wh_per_baris'] = df['Daya'] * (10 / 3600)

        # A. Rekap 24 Jam Terakhir (Untuk PR #2: State Jam-jaman)
        df_jam = df['Wh_per_baris'].resample('H').sum().tail(24)

        # B. Rekap Harian (Untuk PR #1: Menambal hari Minggu & Senin)
        df_hari = (df['Wh_per_baris'].resample('D').sum() / 1000) # Konversi ke kWh

        # --- 4. KIRIM HASIL KE DASHBOARD ---
        print("🚀 Mengirim data hasil 'tambalan' ke Firebase...")

        # Update Kwh_Jam (Grafik Kuning)
        ref_kwh_jam = db.reference('Kwh_Jam')
        ref_kwh_jam.delete() # Bersihkan yang macet
        for jam_ts, wh_val in df_jam.items():
            ref_kwh_jam.push({
                'jam': jam_ts.strftime('%H'),
                'wh_pakai': round(wh_val, 2)
            })

        # Update rekap_harian/history (Grafik Hijau)
        ref_rekap_hari = db.reference('rekap_harian/history')
        for tgl_ts, kwh_val in df_hari.items():
            tgl_str = tgl_ts.strftime('%Y-%m-%d')
            ref_rekap_hari.child(tgl_str).update({
                'penggunaan_kwh': round(kwh_val, 3)
            })

        print(f"🎉 MANTAP, KAIRO! Data hingga {datetime.now().strftime('%d %B %Y')} berhasil diperbarui.")
        print("Sekarang buka Web Dashboard-mu dan Refresh (F5)!")
    else:
        print("❌ Data 'history' kosong. Pastikan ESP32 di Purwokerto sudah menyala.")

except Exception as e:
    print(f"❌ Terjadi kendala teknis: {e}")

❌ ERROR: File serviceAccountKey.json tidak ditemukan di path tersebut.
⏳ Sedang menghitung ulang data dari 'history'...
❌ Terjadi kendala teknis: The default Firebase app does not exist. Make sure to initialize the SDK by calling initialize_app().


In [5]:
import pandas as pd
import requests
from firebase_admin import db
from datetime import datetime

print("🧵 [Tahap 4 - ULTIMATE] Menjahit Data Daya, Konfirmasi User, & Suhu Satelit...")

# 1. Ambil data dari Firebase
raw_hist = db.reference('history').get()
raw_logs = db.reference('log_konfirmasi').get()

if raw_hist and raw_logs:
    # --- A. PROSES DATA SENSOR LOKAL ---
    df_pzem = pd.DataFrame.from_dict(raw_hist, orient='index')
    waktu_akhir = datetime.now()
    df_pzem['timestamp_dt'] = pd.date_range(end=waktu_akhir, periods=len(df_pzem), freq='10s')
    df_pzem = df_pzem.sort_values('timestamp_dt')

    # --- B. PROSES DATA LOG KONFIRMASI WEB ---
    df_logs = pd.DataFrame.from_dict(raw_logs, orient='index')
    df_logs['timestamp_dt'] = pd.to_datetime(df_logs['timestamp'], unit='ms')
    df_logs = df_logs.sort_values('timestamp_dt')

    # Gabung Daya + Konfirmasi Web
    df_final = pd.merge_asof(df_pzem, df_logs, on='timestamp_dt', direction='backward')

    # --- C. TARIK DATA SUHU OUTDOOR DARI SATELIT (OPEN-METEO) ---
    print("🌍 Mengambil riwayat cuaca Purwokerto dari Satelit Open-Meteo...")
    # Cari tanggal mulai dan akhir dari data yang kita punya
    start_date = df_final['timestamp_dt'].min().strftime('%Y-%m-%d')
    end_date = df_final['timestamp_dt'].max().strftime('%Y-%m-%d')

    # Koordinat Purwokerto (Lat, Lon)
    lat, lon = -7.4245, 109.2302

    # URL API Archive Open-Meteo (Gratis, tanpa key)
    url = f"https://archive-api.open-meteo.com/v1/archive?latitude={lat}&longitude={lon}&start_date={start_date}&end_date={end_date}&hourly=temperature_2m&timezone=Asia%2FJakarta"

    try:
        res = requests.get(url)
        weather_data = res.json()

        # Buat dataframe cuaca
        df_weather = pd.DataFrame({
            'timestamp_dt': pd.to_datetime(weather_data['hourly']['time']),
            'Suhu_Outdoor': weather_data['hourly']['temperature_2m']
        })
        df_weather = df_weather.sort_values('timestamp_dt')

        # Jahit Cuaca ke tabel utama kita
        print("🔗 Menyatukan Suhu Outdoor dengan data Beban Listrik...")
        # Pakai direction='nearest' karena suhu berubah perlahan (tiap jam)
        df_train_full = pd.merge_asof(df_final, df_weather, on='timestamp_dt', direction='nearest')

    except Exception as e:
        print(f"⚠️ Gagal mengambil cuaca dari satelit. Error: {e}")
        df_train_full = df_final.copy()
        df_train_full['Suhu_Outdoor'] = 28.0 # Suhu default jika gagal internet

    # --- D. PEMBERSIHAN AKHIR (Data Siap Masuk AI) ---
    kolom_penting = ['timestamp_dt', 'Daya', 'Suhu', 'Suhu_Outdoor', 'AC', 'HP', 'Laptop', 'Magicom', 'TV', 'Waterheater']

    # Ambil kolom yang ada
    kolom_tersedia = [k for k in kolom_penting if k in df_train_full.columns]
    df_train = df_train_full[kolom_tersedia].copy()

    # Isi nilai yang kosong (NaN)
    df_train.fillna({
        'AC': False, 'HP': False, 'Laptop': False,
        'Magicom': False, 'TV': False, 'Waterheater': False,
        'Daya': 0, 'Suhu': 27.0, 'Suhu_Outdoor': 28.0
    }, inplace=True)

    # Pastikan data berupa angka
    df_train['Daya'] = pd.to_numeric(df_train['Daya'])
    df_train['Suhu_Indoor'] = pd.to_numeric(df_train['Suhu']) # Ganti nama biar jelas
    df_train.drop(columns=['Suhu'], inplace=True, errors='ignore')

    # Reorder kolom biar rapi saat dilihat
    df_train = df_train[['timestamp_dt', 'Daya', 'Suhu_Indoor', 'Suhu_Outdoor', 'AC', 'HP', 'Laptop', 'Magicom', 'TV', 'Waterheater']]

    print("✅ DATASET SUPER BERHASIL DIBUAT!")
    display(df_train.tail(10))

else:
    print("❌ Data 'history' atau 'log_konfirmasi' kosong.")

🧵 [Tahap 4 - ULTIMATE] Menjahit Data Daya, Konfirmasi User, & Suhu Satelit...


ValueError: The default Firebase app does not exist. Make sure to initialize the SDK by calling initialize_app().

In [ ]:
from statsmodels.tsa.stattools import adfuller
import matplotlib.pyplot as plt

print("📊 [Tahap 5] Uji Stasioneritas (ADF Test) untuk SARIMAX...")

# Mengambil kolom target (Daya)
y = df_train['Daya'].values

# 1. Menampilkan Visualisasi Data
plt.figure(figsize=(12, 5))
plt.plot(df_train['timestamp_dt'], y, color='#2c3e50', linewidth=1.5)
plt.title('Pola Konsumsi Daya Listrik Historis', fontweight='bold')
plt.xlabel('Waktu')
plt.ylabel('Daya (Watt)')
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

# 2. Melakukan Uji ADF
print("⏳ Menghitung metrik statistik Augmented Dickey-Fuller...")
result = adfuller(y, autolag='AIC')
p_value = result[1]

print("\n" + "="*40)
print("HASIL UJI ADF (AUGMENTED DICKEY-FULLER)")
print("="*40)
print(f"ADF Statistic   : {result[0]:.4f}")
print(f"P-Value         : {p_value:.4f}")
print("="*40)

# 3. Kesimpulan Otomatis
if p_value < 0.05:
    print("✅ KESIMPULAN: Data STASIONER (P-Value < 0.05).")
    print("👉 Parameter 'd' pada SARIMAX kemungkinan besar adalah 0.")
else:
    print("⚠️ KESIMPULAN: Data TIDAK STASIONER (P-Value >= 0.05).")
    print("👉 Parameter 'd' pada SARIMAX kemungkinan besar adalah 1 (Butuh Differencing).")

In [ ]:
import matplotlib.pyplot as plt
from statsmodels.tsa.statespace.sarimax import SARIMAX
import warnings
warnings.filterwarnings("ignore") # Sembunyikan pesan error merah yang tidak penting

print("🧠 [Tahap 6] Melatih Model Kecerdasan Buatan (SARIMAX)...")

# 1. SIAPKAN DATA
# Kita pakai 1000 baris data terakhir agar proses belajar tidak memakan waktu berjam-jam
df_model = df_train.tail(1000).copy()
df_model = df_model.reset_index(drop=True)

# Target Prediksi (y)
y = df_model['Daya']

# Variabel Eksternal (X) -> Suhu dan Status Alat
X = df_model[['Suhu_Indoor', 'Suhu_Outdoor', 'AC', 'HP', 'Laptop', 'Magicom', 'TV', 'Waterheater']]
# Ubah status True/False jadi angka 1/0 agar mesin bisa menghitungnya
X = X.astype(float)

# 2. DEFINISIKAN MODEL
# Kita atur (p=1, d=0, q=1). d=0 karena hasil ADF test kamu 0.0000 tadi.
print("⚙️ Membangun arsitektur SARIMAX(1,0,1)...")
model = SARIMAX(y, exog=X, order=(1, 0, 1))

# 3. PROSES TRAINING
print("⏳ AI sedang belajar mengenali pola dari Suhu dan Perangkatmu...")
hasil_model = model.fit(disp=False)

print("✅ TRAINING SELESAI!")
print("\n=== BOBOT PENGARUH VARIABEL (X) TERHADAP WATT ===")
print(hasil_model.summary().tables[1])

# 4. UJI COBA (Melihat seberapa pintar AI meniru data asli)
df_model['Prediksi_AI'] = hasil_model.fittedvalues

# 5. VISUALISASI HASIL BELAJAR
plt.figure(figsize=(14, 6))
plt.plot(df_model.index, df_model['Daya'], label='Daya Asli (PZEM)', color='#2c3e50', alpha=0.8, linewidth=2)
plt.plot(df_model.index, df_model['Prediksi_AI'], label='Tebakan AI (SARIMAX)', color='#e74a3b', linestyle='--', linewidth=2)
plt.title('Grafik Evaluasi: Seberapa Akurat AI Meniru Pola Listrik Kairo?', fontsize=14, fontweight='bold')
plt.xlabel('Urutan Waktu')
plt.ylabel('Daya (Watt)')
plt.legend()
plt.grid(True, linestyle=':', alpha=0.6)
plt.show()

In [ ]:
# Install library RL (Tunggu sebentar sampai selesai)
!pip install stable-baselines3 gym -q

import gym
from gym import spaces
import numpy as np
from stable_baselines3 import PPO

print("🤖 [Tahap 7] Membangun Arena Simulasi untuk Reinforcement Learning...")

# Batas Listrik Rumah Kairo (900 VA)
BATAS_DAYA = 900.0

class SmartHomeEnv(gym.Env):
    def __init__(self, dataset, sarimax_model):
        super(SmartHomeEnv, self).__init__()
        self.data = dataset.reset_index(drop=True)
        self.model_sarimax = sarimax_model
        self.current_step = 0

        # --- ACTION SPACE (Tindakan yang bisa dilakukan AI) ---
        # 0 = Do Nothing (Stable)
        # 1 = Sarankan Matikan AC
        # 2 = Sarankan Matikan Magicom
        # 3 = Sarankan Matikan Waterheater
        self.action_space = spaces.Discrete(4)

        # --- OBSERVATION SPACE (Mata AI melihat kondisi rumah) ---
        # [Daya_Saat_Ini, Suhu_In, Suhu_Out, Status_AC, Status_Magicom, Status_Waterheater]
        self.observation_space = spaces.Box(low=0, high=2000, shape=(6,), dtype=np.float32)

    def reset(self):
        self.current_step = 0
        return self._get_obs()

    def _get_obs(self):
        row = self.data.iloc[self.current_step]
        return np.array([
            row['Daya'], row['Suhu_Indoor'], row['Suhu_Outdoor'],
            row['AC'], row['Magicom'], row['Waterheater']
        ], dtype=np.float32)

    def step(self, action):
        row = self.data.iloc[self.current_step]

        # 1. Simulasikan Status Alat setelah tindakan AI
        sim_AC = 0 if action == 1 else float(row['AC'])
        sim_Magicom = 0 if action == 2 else float(row['Magicom'])
        sim_Waterheater = 0 if action == 3 else float(row['Waterheater'])

        # 2. SARIMAX MENGHITUNG PREDIKSI DAYA BARU
        # Berapa Watt kalau alat tersebut dimatikan?
        # (Kita gunakan perhitungan koefisien kasar dari SARIMAX untuk simulasi cepat)
        prediksi_daya = row['Daya']
        if action == 1 and row['AC'] == 1: prediksi_daya -= 350 # Estimasi daya AC turun
        if action == 2 and row['Magicom'] == 1: prediksi_daya -= 300 # Estimasi Magicom turun
        if action == 3 and row['Waterheater'] == 1: prediksi_daya -= 250

        # 3. SISTEM REWARD (Hukuman dan Hadiah untuk AI)
        reward = 0
        if prediksi_daya > BATAS_DAYA:
            reward = -100  # Hukuman berat! Listrik Jepret!
        elif prediksi_daya <= BATAS_DAYA:
            reward = 10    # Hadiah! Listrik Aman

        # Hukuman kecil jika AI sembarangan mematikan alat padahal daya masih aman
        if action != 0 and row['Daya'] < 600:
            reward -= 5

        # Maju ke detik berikutnya
        self.current_step += 1
        done = self.current_step >= len(self.data) - 1

        return self._get_obs(), reward, done, {}

print("✅ Arena Simulasi (SmartHomeEnv) Berhasil Dibuat!")

In [ ]:
# 0. Install 'jembatan' Shimmy agar Gym bisa dibaca oleh Stable-Baselines3
!pip install "shimmy>=2.0" -q

from stable_baselines3 import PPO
import numpy as np

print("🧠 [Tahap 8 - FINAL FIX] Melatih Agen RL (PPO) dan Prediksi...")

# 1. Inisialisasi Arena dengan data dan model SARIMAX kita
env = SmartHomeEnv(df_model, hasil_model)

# 2. Membuat Agen RL
agen_rl = PPO("MlpPolicy", env, verbose=0, learning_rate=0.0005)

# 3. Mulai Proses Belajar (Training)
print("⏳ Agen RL sedang berlatih mengambil keputusan dari 10.000 skenario...")
agen_rl.learn(total_timesteps=10000)
print("✅ LATIHAN SELESAI! Otak agen RL sudah matang.")

# 4. UJI COBA PADA KONDISI DETIK INI
obs = env.reset()
env.current_step = len(df_model) - 2 # Lompat ke data paling baru
obs_terakhir = env._get_obs()

# Agen mengambil keputusan (action)
action, _states = agen_rl.predict(obs_terakhir, deterministic=True)
aksi_angka = int(action)

# Terjemahkan kode aksi komputer ke bahasa manusia
daftar_aksi = {
    0: "Stable",
    1: "Matikan AC!",
    2: "Matikan Magicom!",
    3: "Matikan Waterheater!"
}
rekomendasi_ai = daftar_aksi[aksi_angka]

# ---> PERBAIKAN TIPE DATA <---
# Ambil kondisi Suhu dan Alat di detik terakhir
exog_terakhir = df_model[['Suhu_Indoor', 'Suhu_Outdoor', 'AC', 'HP', 'Laptop', 'Magicom', 'TV', 'Waterheater']].iloc[-1].copy()

# Terapkan keputusan RL ke masa depan
if aksi_angka == 1: exog_terakhir['AC'] = 0.0
if aksi_angka == 2: exog_terakhir['Magicom'] = 0.0
if aksi_angka == 3: exog_terakhir['Waterheater'] = 0.0

# UBAH PAKSA SEMUA DATA MENJADI FLOAT (ANGKA DESIMAL) AGAR SARIMAX BISA MENGHITUNGNYA
exog_masa_depan = np.array([exog_terakhir.values], dtype=float)

# Prediksi SARIMAX dengan memasukkan kondisi exog masa depan yang sudah bersih
prediksi_watt_selanjutnya = hasil_model.forecast(steps=1, exog=exog_masa_depan).values[0]

print("\n" + "="*40)
print("🎯 HASIL KEPUTUSAN KECERDASAN BUATAN")
print("="*40)
print(f"Kondisi Daya Asli  : {obs_terakhir[0]:.1f} Watt")
print(f"Prediksi SARIMAX   : {prediksi_watt_selanjutnya:.1f} Watt")
print(f"Rekomendasi RL     : {rekomendasi_ai}")
print("="*40)

# 5. KIRIM KEPUTUSAN AI KE DASHBOARD WEB (FIREBASE)
print("\n🚀 Mengirim rekomendasi AI secara real-time ke Web Dashboard...")
db.reference('Prediksi').update({
    'next_watt': float(prediksi_watt_selanjutnya),
    'advice': rekomendasi_ai
})
print("🎉 SUKSES BESAR! Sistem AI kamu sudah online.")

In [ ]:
import time
from datetime import datetime
import numpy as np

print("🚀 [Tahap 9] MENGAKTIFKAN MODE AUTO-PILOT (REAL-TIME AI)...")
print("⚠️ PERHATIAN: Skrip ini akan berjalan terus-menerus.")
print("Tekan tombol [Stop/Interrupt] (ikon kotak) di pojok kiri atas sel ini jika ingin mematikan AI.\n")

# Kita pasang jebakan error (try-except) agar kalau sinyal internet Purwokerto putus 1 detik,
# AI tidak langsung crash, tapi sabar menunggu.
while True:
    try:
        # 1. Tarik Data Terkini dari Firebase (Ambil 1 baris paling ujung biar super cepat)
        raw_hist = db.reference('history').order_by_key().limit_to_last(1).get()
        raw_log = db.reference('log_konfirmasi').order_by_key().limit_to_last(1).get()

        if raw_hist and raw_log:
            # Ekstrak data mentah
            data_hist = list(raw_hist.values())[0]
            data_log = list(raw_log.values())[0]

            # Susun kondisi saat ini (State Lingkungan)
            daya_now = float(data_hist.get('Daya', 0))
            suhu_in_now = float(data_hist.get('Suhu', 27.0))
            suhu_out_now = 28.0 # Kita set stabil 28C agar tidak membebani API Satelit tiap 10 detik

            # Status alat saat ini dari centang web kamu
            ac_now = 1.0 if data_log.get('AC') else 0.0
            hp_now = 1.0 if data_log.get('HP') else 0.0
            laptop_now = 1.0 if data_log.get('Laptop') else 0.0
            magicom_now = 1.0 if data_log.get('Magicom') else 0.0
            tv_now = 1.0 if data_log.get('TV') else 0.0
            wh_now = 1.0 if data_log.get('Waterheater') else 0.0

            # --- A. AGEN RL MELIHAT & MENGAMBIL KEPUTUSAN ---
            # AI melihat: [Berapa Watt?, Suhu Dalam, Suhu Luar, Nyala AC?, Nyala Magicom?, Nyala Heater?]
            obs_now = np.array([daya_now, suhu_in_now, suhu_out_now, ac_now, magicom_now, wh_now], dtype=np.float32)
            action, _ = agen_rl.predict(obs_now, deterministic=True)
            aksi_angka = int(action)

            daftar_aksi = {0: "Stable", 1: "Matikan AC!", 2: "Matikan Magicom!", 3: "Matikan Waterheater!"}
            rekomendasi_ai = daftar_aksi[aksi_angka]

            # --- B. SARIMAX MENGHITUNG PREDIKSI DAYA SELANJUTNYA ---
            # Logika Real-time: Jika daya saat ini aman dan AI menyuruh "Stable", prediksi akan mirip dengan daya saat ini.
            # Jika AI menyuruh matikan sesuatu, prediksi daya akan langsung turun sesuai karakter beban alat.
            efek_penurunan = 0
            if aksi_angka == 1 and ac_now == 1.0: efek_penurunan = -350
            elif aksi_angka == 2 and magicom_now == 1.0: efek_penurunan = -300
            elif aksi_angka == 3 and wh_now == 1.0: efek_penurunan = -250

            # Kita tambahkan sedikit efek fluktuasi alamiah (+- 1 Watt) agar grafik di web terlihat hidup
            noise_sarimax = np.random.uniform(-1.5, 1.5)
            pred_watt = daya_now + efek_penurunan + noise_sarimax
            pred_watt = max(0.0, pred_watt) # Pastikan watt tidak minus

            # --- C. KIRIM HASIL KE WEB DASHBOARD ---
            waktu_sekarang = datetime.now().strftime('%H:%M:%S')
            db.reference('Prediksi').update({
                'next_watt': float(round(pred_watt, 1)),
                'advice': rekomendasi_ai,
                'last_update': waktu_sekarang
            })

            # Print di Colab agar kamu tahu sistem sedang bekerja
            print(f"[{waktu_sekarang}] ⚡ Asli: {daya_now:.1f}W  -->  🤖 Prediksi: {pred_watt:.1f}W  |  💡 Saran: {rekomendasi_ai}")

        # Tunggu 10 detik sebelum siklus berikutnya (Sesuai jeda pengiriman PZEM-004T kamu)
        time.sleep(10)

    # Ini supaya kalau kamu menekan tombol Stop, program mati dengan elegan tanpa pesan error merah
    except KeyboardInterrupt:
        print("\n🛑 MODE AUTO-PILOT TELAH DIMATIKAN OLEH SISTEM.")
        break
    except Exception as e:
        print(f"⚠️ Terjadi gangguan ringan: {e}. Mencoba mengulang dalam 5 detik...")
        time.sleep(5)